# Phase 4: Sequence Construction & Chronological Partitioning

**Objective**: Convert windowed feature vectors into temporal sequences:
$$X = [X_1, X_2, \dots, X_T], \quad X_t \in \mathbb{R}^D$$
Sequences preserve strict chronological ordering. No window shuffling is allowed.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np

from src.sequences import (
    build_sequences_from_features,
    SequenceNormaliser,
    split_sequences_chronologically,
)

feat_df = pd.read_csv('../data/processed/features_all.csv')
print(f"Loaded {len(feat_df)} feature rows.")

## 1. Sequence Assembly per Trip

In [ ]:
sequences, lengths, meta = build_sequences_from_features(feat_df)
print(f"Constructed {len(sequences)} trip sequences.")
print(f"Sample sequence shape: {sequences[0].shape} (Time windows x Features)")

## 2. Chronological Train / Val / Test Partitioning
Split recordings chronologically (70% train, 15% validation, 15% test) with gap buffer between splits.

In [ ]:
(train_seqs, val_seqs, test_seqs,
 train_meta, val_meta, test_meta) = split_sequences_chronologically(
    sequences, meta, train_frac=0.70, val_frac=0.15, gap=1
)

print(f"Train sequences: {len(train_seqs)} ({sum(len(s) for s in train_seqs)} windows)")
print(f"Val sequences:   {len(val_seqs)} ({sum(len(s) for s in val_seqs)} windows)")
print(f"Test sequences:  {len(test_seqs)} ({sum(len(s) for s in test_seqs)} windows)")

## 3. Normalization without Data Leakage

In [ ]:
normaliser = SequenceNormaliser()
train_norm = normaliser.fit_transform(train_seqs)
val_norm = normaliser.transform(val_seqs)
test_norm = normaliser.transform(test_seqs)

print("Normalization complete. Feature scaler fitted strictly on training sequences.")